<a href="https://colab.research.google.com/github/MarcoLuw/QoE-MOS-Prediction/blob/main/model_training/xgboost_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Feature description

| # | Column | Category | Meaning / Practical Use | Values / Details from README |
|---|--------|----------|--------------------------|------------------------------|
| 1 | id | Identifier | Unique identifier of the sample (one video-viewing test record). | Not further defined in README |
| 2 | user_id | Identifier | Identifier of the tester who performed the session; links multiple samples to the same person. | Not further defined in README |
| 3 | QoA_VLCresolution | Video (QoA) | The actual picture resolution at which the video was played in VLC during the session, i.e. how many vertical lines of detail were rendered on screen. In practice it tells you whether the user watched a lower-detail or higher-detail version of the clip. | README notes resolution is a class identifier with values 240p and 360p |
| 4 | QoA_VLCbitrate | Video (QoA) | The video bitrate reported by VLC, i.e. the amount of video data per second during playback; higher values mean more data and generally better image quality. | Not further defined in README |
| 5 | QoA_VLCframerate | Video (QoA) | The playback frame rate reported by VLC, i.e. how many video frames per second were shown; relates to motion smoothness. | Not further defined in README |
| 6 | QoA_VLCdropped | Video (QoA) | The count of video frames dropped by VLC during playback; a measure of rendering/decoding loss experienced by the user. | Not further defined in README |
| 7 | QoA_VLCaudiorate | Video (QoA) | The audio bitrate/rate reported by VLC for the session's sound track. | Not further defined in README |
| 8 | QoA_VLCaudioloss | Video (QoA) | The amount of audio lost during playback as reported by VLC; relates to sound dropouts. | Not further defined in README |
| 9 | QoA_BUFFERINGcount | Video (QoA) | The number of buffering (stalling) events that occurred during the video session; how many times playback paused to rebuffer. | Not further defined in README |
| 10 | QoA_BUFFERINGtime | Video (QoA) | The total time spent buffering/stalling during the session; the cumulative duration playback was interrupted. | Not further defined in README |
| 11 | QoS_type | Network (QoS) | The mobile network access technology used during the test. | Coded: 1=EDGE, 2=UMTS, 3=HSPA, 4=HSPAP, 5=LTE |
| 12 | QoS_operator | Network (QoS) | The French mobile network operator used for the session. | Coded: 1=SFR, 2=BOUYGUES, 3=ORANGE, 4=FREE |
| 13 | QoD_model | Device (QoD) | The model of the device (phone or tablet) used by the tester. | Not further defined in README |
| 14 | QoD_os-version | Device (QoD) | The Android OS version running on the test device. | Not further defined in README |
| 15 | QoD_api-level | Device (QoD) | The Android API level of the device (the platform API number associated with the OS version). | Not further defined in README |
| 16 | QoU_sex | User (QoU) | The tester's gender. | Coded: 0=Woman, 1=Man |
| 17 | QoU_age | User (QoU) | The tester's age. | README states testers were aged 19 to 38 |
| 18 | QoU_study | User (QoU) | The tester's highest education level. | Coded: 5=University, 4=Secondary school, 3=College, 2=Primary school, 1=Other |
| 19 | QoF_begin_time | Feedback (QoF) | The begin time of the video session (when the viewing started). | Not further defined in README |
| 20 | QoF_shift | Feedback (QoF) | A shift value related to the session. | Not further defined in README |
| 21 | QoF_audio | Feedback (QoF) | User feedback relating to the audio of the session. | Not further defined in README |
| 22 | QoF_video | Feedback (QoF) | User feedback relating to the video of the session. | Not further defined in README |
| 23 | MOS | Target / Score | Mean Opinion Score the tester gives at the end of each video view; the subjective quality rating used as the label. | Coded: 5=Excellent, 4=Good, 3=Fair, 2=Poor, 1=Bad |

# Download dataset

GitHub Link: https://github.com/Lamyne/Poqemon-QoE-Dataset

In [2]:
!wget https://raw.githubusercontent.com/Lamyne/Poqemon-QoE-Dataset/master/pokemon.csv -O ~/pokemon.csv
# !wget https://raw.githubusercontent.com/andreaaraldo/machine-learning-for-networks/master/course_library/visualization.py

--2026-11-09 14:00:44--  https://raw.githubusercontent.com/Lamyne/Poqemon-QoE-Dataset/master/pokemon.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8000::154, 2606:50c0:8002::154, 2606:50c0:8003::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8000::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 158675 (155K) [text/plain]
Saving to: ‘/mnt/hdd/homes/hlu/pokemon.csv’

/mnt/hdd/homes/hlu/ 100%[===================>] 154.96K  --.-KB/s    in 0.04s   

2026-11-09 14:00:45 (3.88 MB/s) - ‘/mnt/hdd/homes/hlu/pokemon.csv’ saved [158675/158675]



In [3]:
!ls

random_forest.ipynb  xgboost_regression.ipynb


# Python package import

If XGBoost is not installed, run: `!pip install xgboost`

In [ ]:
# !uv pip install xgboost

In [ ]:
import pandas as pd
import numpy as np
# import visualization
import matplotlib.pyplot as plt
from pandas.plotting import scatter_matrix

from sklearn.preprocessing import PowerTransformer, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# The following is to be able to mount Google Drive
# from google.colab import drive
import pickle # To save the model
from os.path import isfile
from pathlib import Path
from datetime import datetime

# Artifact location

In [ ]:
model_tag = "XGB-with-QoA-QoU-QoD-model"
model_artifact = f'{model_tag}.pkl'

# Timestamp for this run, to avoid overwriting previous runs
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Always under home, in a dedicated results folder
SAVE_PATH = Path.home() / "ds_result" / timestamp
SAVE_PATH.mkdir(parents=True, exist_ok=True)

In [ ]:
CAT_FEATURES = ["QoD_model"]
NUM_FEATURES = ["QoA_VLCresolution", "QoA_VLCbitrate", "QoA_VLCframerate",
            "QoA_VLCdropped", "QoA_VLCaudiorate", "QoA_VLCaudioloss",
            "QoA_BUFFERINGcount", "QoA_BUFFERINGtime", "QoU_sex", "QoU_age", "QoU_Ustedy"]
FEATURES = CAT_FEATURES + NUM_FEATURES
TARGET = "MOS"

In [7]:
df = pd.read_csv("~/pokemon.csv")

In [8]:
df.head()

,id,user_id,QoA_VLCresolution,QoA_VLCbitrate,QoA_VLCframerate,QoA_VLCdropped,QoA_VLCaudiorate,QoA_VLCaudioloss,QoA_BUFFERINGcount,QoA_BUFFERINGtime,...,QoD_os-version,QoD_api-level,QoU_sex,QoU_age,QoU_Ustedy,QoF_begin,QoF_shift,QoF_audio,QoF_video,MOS
0,590,71,360,528.39294,24.950000,0,43.800000,0,2,683,...,4.1.1(122573.16),16,1,20,5,3,5,3,4,3
1,428,46,360,402.64664,30.066667,0,44.200000,0,2,690,...,4.4.2(I9195XXUCNK1),19,1,25,5,4,5,5,5,5
2,406,44,360,735.84070,24.200000,0,44.183333,0,2,840,...,4.1.2(I9300XXELL4),16,1,22,5,3,5,4,4,4
3,1256,133,240,529.48830,24.116667,9,43.850000,0,2,868,...,4.4.4(suv3Rw),19,1,31,5,4,5,5,5,5
4,244,22,360,736.00085,24.066667,0,43.850000,0,2,869,...,4.1.2(I9300XXELL4),16,0,26,5,5,5,4,5,5


# Data Preprocessing

- Remove duplications
- Remove error values
- Handle missing values

In [9]:
def preprocess(df):
  df = df.drop_duplicates(keep="first") # drop duplications
  df = df[df["QoA_VLCresolution"] != 16]  # drop error values
  df = df.dropna(subset=FEATURES + [TARGET]) # drop missing values
  df = df.reset_index(drop=True)

  X = df[FEATURES]
  y = df[TARGET]
  return X, y

# Prepare data

In [10]:
def prepare_data(X, y):
  X_train, X_test, y_train, y_test = train_test_split(
      X, y, test_size=0.2, random_state=42
  )

  # feature engineering: transform (de-skew) - no need for tree-based models
  # pt = PowerTransformer(method="yeo-johnson")
  # X_train = pt.fit_transform(X_train)
  # X_test = pt.transform(X_test)

  # scale
  scaler = StandardScaler()
  X_train = scaler.fit_transform(X_train)
  X_test = scaler.transform(X_test)

  return X_train, X_test, y_train, y_test, scaler

In [ ]:
# Prepare data with one hot encoding
def prepare_data_with_ohe(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # one-hot encode the categorical column
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    train_cat = ohe.fit_transform(X_train[CAT_FEATURES])
    test_cat  = ohe.transform(X_test[CAT_FEATURES])
    cat_names = ohe.get_feature_names_out(CAT_FEATURES)

    # scale the numeric columns
    scaler = StandardScaler()
    train_num = scaler.fit_transform(X_train[NUM_FEATURES])
    test_num  = scaler.transform(X_test[NUM_FEATURES])

    # combine numeric + encoded columns
    X_train = np.hstack([train_num, train_cat])
    X_test  = np.hstack([test_num,  test_cat])

    # full feature-name list (numeric first, then device dummies)
    encoded_names = list(NUM_FEATURES) + list(cat_names)

    return X_train, X_test, y_train, y_test, scaler, ohe, encoded_names

In [ ]:
X, y = preprocess(df)
# X_train, X_test, y_train, y_test, scaler = prepare_data(X, y)

# Split with one-hot encoding
X_train, X_test, y_train, y_test, scaler, ohe, encoded_features = prepare_data_with_ohe(X, y)

In [ ]:
encoded_features

# Training an XGBoost model

## Define grid params

In [12]:
# number of boosting rounds (trees)
n_estimators = [100, 300, 500]

# max depth of a tree
max_depth = [3, 5, 7, 10]

# learning rate (shrinkage) - XGBoost specific, key knob
learning_rate = [0.01, 0.05, 0.1, 0.3]

# fraction of samples used per tree (row subsampling)
subsample = [0.6, 0.8, 1.0]

# fraction of features used per tree (column subsampling)
colsample_bytree = [0.6, 0.8, 1.0]

# minimum loss reduction to make a split (regularization, like pruning)
gamma = [0, 0.1, 0.5, 1]

# minimum sum of instance weight needed in a child
min_child_weight = [1, 3, 5]

# L2 regularization term on weights
reg_lambda = [0.1, 1, 10]

# L1 regularization term on weights
reg_alpha = [0, 0.1, 1]

grid_param = {
    'n_estimators': n_estimators,
    'max_depth': max_depth,
    'learning_rate': learning_rate,
    'subsample': subsample,
    'colsample_bytree': colsample_bytree,
    'gamma': gamma,
    'min_child_weight': min_child_weight,
    'reg_lambda': reg_lambda,
    'reg_alpha': reg_alpha,
}

## Random Grid Search

The grid above spans a large number of combinations x 5 k-fold, a very large computational expense.

`RandomizedSearchCV` helps to overcome the hurdle by randomly sampling a specified number of possible candidate models.

For example, only randomly sample 200 models.

In [13]:
# The search parallelizes across 200x5 fits (all CPUs) (n_jobs = -1)
# while each individual model trains single-threaded (n_jobs = 1)
regressor = XGBRegressor(random_state=4, n_jobs=1, objective='reg:squarederror')
search = RandomizedSearchCV(
                            scoring='neg_root_mean_squared_error',
                            estimator=regressor,
                            param_distributions=grid_param,
                            n_iter=50, # We just test 200 configurations
                            verbose=2,
                            random_state=42, # Random state of GridSearch only
                            n_jobs=-1, # Train models in parallel using all CPUs
                            cv=5 # 5-fold validation
                          )

In [ ]:
search.fit(X_train, y_train)

## Get best model

In [15]:
print(search.best_params_)
best_model = search.best_estimator_

{'subsample': 0.6, 'reg_lambda': 10, 'reg_alpha': 0.1, 'n_estimators': 500, 'min_child_weight': 1, 'max_depth': 7, 'learning_rate': 0.01, 'gamma': 1, 'colsample_bytree': 1.0}


With this best model, we try to (re)train it on the whole dataset

In [16]:
best_model.fit(X_train, y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,1.0
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


# Evaluating Model

In [ ]:
y_pred = best_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Save metrics to CSV
results = pd.DataFrame({
    "Metric": ["RMSE", "MAE", "R2"],
    "Score": [rmse, mae, r2]
})

results.to_csv(SAVE_PATH / "evaluation_results.csv", index=False)

print(results)

In [ ]:
# Predicted vs actual MOS
title = "Predicted vs Actual MOS"
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.4)

# diagonal reference line (perfect prediction)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, 'r--')

plt.xlabel("Actual MOS")
plt.ylabel("Predicted MOS")
plt.title(title)
plt.savefig(SAVE_PATH / "pred_vs_actual.png", dpi=150, bbox_inches="tight")  # save
plt.show()

# Model Explanation

## Feature importance (impurity-based)

In [ ]:
importances = pd.Series(
    best_model.feature_importances_,
    # index=FEATURES
    index=encoded_features
).sort_values(ascending=False)

print(importances)

In [ ]:
# Frequency per value of a column (check for high cardinality feature)
df["QoA_BUFFERINGtime"].value_counts().sort_index()

In [ ]:
df["QoA_VLCframerate"].value_counts().sort_index()

We observe that:
- `QoA_BUFFERINGtime` and `QoA_VLCframerate` have high feature importance weights (0.579337 and 0.131076).
- `QoA_BUFFERINGtime` and `QoA_VLCframerate` are also high cardinality features (1327 and 309 unique values)

Based on the [feature importance document](https://scikit-learn.org/stable/auto_examples/ensemble/plot_forest_importances.html#feature-importance-based-on-mean-decrease-in-impurity), impurity-based feature importances can be misleading for high cardinality features. 

Therefore, we are not confident that the two top-weighted features are important for predicting MOS.

=> We will alternatively use [Permutation feature importance](https://scikit-learn.org/stable/modules/permutation_importance.html#permutation-importance)

In [ ]:
importances.to_csv(SAVE_PATH / "importance_builtin.csv")  # save built-in importance table

In [ ]:
importances.sort_values().plot(kind="barh", figsize=(8, 5))
plt.xlabel("Importance (impurity reduction)")
plt.title("XGBoost feature importance")
plt.tight_layout()
plt.savefig(SAVE_PATH / "importance_builtin.png", dpi=150, bbox_inches="tight")  # save plot
plt.show()

## Permutation importance

Permutation feature importance reflects how important a feature is for a particular model

In this case, we use `scoring=rmse` for a regression model

In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    best_model, X_test, y_test,
    scoring="neg_root_mean_squared_error",
    n_repeats=20,
    random_state=42,
    n_jobs=-1,
)

perm = pd.Series(
    result.importances_mean,
    # index=FEATURES
    index=encoded_features
).sort_values(ascending=False)

print(perm)

From the table above, we observe that `QoA_BUFFERINGtime` and `QoA_VLCframerate` are actually important for the model
- `QoF_begin` and `QoF_shift` are secondarily important, but since our objective is to predict the MOS based on the network condition, these two user feedback features, together with `QoF_video` and `QoF_audio` will be removed due to their data leakage and irrelevance
- `QoA_BUFFERINGtime` has higher importance than `QoA_BUFFERINGcount` -> total duration of buffering (user waiting time) provides more useful predictive information
- User profile features somehow have higher average importance than video parameters and network information -> Diversity in user background could have a more substantial impact on MOS scores (their experiences and perspectives). Though both user and network profiles seem to have little effect on MOS prediction

In [ ]:
# save permutation importance table

# perm_df = pd.DataFrame({
#     "importance_mean": result.importances_mean,
#     "importance_std": result.importances_std,
# }, index=FEATURES).sort_values("importance_mean", ascending=False)
# perm_df.to_csv(SAVE_PATH / "importance_permutation.csv")

perm_df = pd.DataFrame({
    "importance_mean": result.importances_mean,
    "importance_std": result.importances_std,
}, index=encoded_features).sort_values("importance_mean", ascending=False)
perm_df.to_csv(SAVE_PATH / "importance_permutation.csv")

In [ ]:
sorted_idx = result.importances_mean.argsort()

plt.figure(figsize=(8, 5))
plt.barh(
    # np.array(FEATURES)[sorted_idx],
    np.array(encoded_features)[sorted_idx],
    result.importances_mean[sorted_idx],
    xerr=result.importances_std[sorted_idx],
)
plt.xlabel("Drop in score when feature is shuffled")
plt.title("Permutation importance (test set)")
plt.tight_layout()
plt.savefig(SAVE_PATH / "importance_permutation.png", dpi=150, bbox_inches="tight")  # save plot
plt.show()

## SHAP

So far, we have known how importance each feature are thanks to both built-in (impurity-based) and permutation feature importances.
But they lack the direction of the importance, for example, whether higher BufferingTime provides higher or lower MOS.

=> SHAP can help: giving direction and per-sample effects to the model

In [ ]:
import shap

# Build the explainer
explainer = shap.TreeExplainer(best_model)
X_test_df_shap = pd.DataFrame(
                X_test, 
                # columns=FEATURES
                columns=encoded_features
            )
shap_values = explainer.shap_values(X_test_df_shap)

In [ ]:
shap.summary_plot(shap_values, X_test_df_shap, plot_type="bar")

In [ ]:
# Save and show SHAP beeswarm plot
shap.summary_plot(shap_values, X_test_df_shap, show=False)
plt.tight_layout()
plt.savefig(SAVE_PATH / "shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

## Aggregated importance of categorical columns

We group those sub-features (generated by OHE) together and **sum their importance values** to get the score for the original categorical variable

Built-in, permutation importance and SHAP mean on `QoD_model_*`

In [ ]:
# aggregated_importances = pd.Dataframe(columns=["built-in", "permutation", "SHAP mean |value|"], index=CAT_FEATURES)
aggregated_importances = pd.DataFrame(
    columns=["built-in", "permutation", "SHAP mean |value|"],
    index=CAT_FEATURES
)
perm_series = pd.Series(result.importances_mean, index=encoded_features)

for col in CAT_FEATURES:
    col_prefix = f'{col}_'
    
    # Built-in and permutation importance
    device_cols = [c for c in encoded_features if c.startswith(col_prefix)]
    builtin_val = importances[device_cols].sum()
    perm_val = perm_series[device_cols].sum()
    
    # SHAPE
    # indices of the device one-hot columns
    device_idx = [i for i, c in enumerate(encoded_features) if c.startswith(col_prefix)]
    # sum SHAP values across all device columns, per row -> one "device" SHAP value per sample
    device_shap = shap_values[:, device_idx].sum(axis=1)
    # overall importance of QoD_model (mean absolute SHAP, like the bar plot uses)
    shap_val = np.abs(device_shap).mean()
    
    print(f"{col} total (built-in):", builtin_val)
    print(f"{col} total (permutation):", perm_val)
    print(f"{col} total (SHAP, mean |value|):", shap_val)

    # Add to the unified df
    aggregated_importances.loc[col] = {
        "built-in": builtin_val,
        "permutation": perm_val,
        "SHAP mean |value|": shap_val,
    }

# Save aggregated_importances df
aggregated_importances.to_csv(SAVE_PATH / "aggregated_importances_cate_features.csv")

# Save model

## On Google Colab

In [19]:
# mount_point = '/content/drive'
# drive.mount(mount_point, force_remount=True)
# drive_path = mount_point + '/My Drive/'

# save_path = drive_path + 'Colab Notebooks/TSP - Data Science/Project/Models/'
# filename = save_path + f'RF.pkl'
# pickle.dump(best_model, open(filename, 'wb'))

In [ ]:
%cd $save_path
!ls

Optional: create a function to reduce duplication

In [21]:
# def search_or_load(model_filename, search, X_train, y_train):
#   if (isfile(model_filename)):
#     print("Loading model")
#     model = pickle.load(open(model_filename, 'rb'))

#   else:
#     print("Searching the best hyper_parameters")
#     search.fit(X_train, y_train)
#     print(search.best_params_)
#     model = search.best_estimator_
#     print("Training model")
#     model.fit(X_train,y_train)
#     pickle.dump(model, open(model_filename, 'wb'))
#     print("Model saved in in file ", model_filename)
#   return model

## On Linux

In [ ]:
FINAL_MODEL_PATH = str(SAVE_PATH / model_artifact)

pickle.dump(best_model, open(FINAL_MODEL_PATH, 'wb'))

print("Final model path:", FINAL_MODEL_PATH)